```{index} console; input; output
```

(io)=

# Input and Output


```{contents}
:local:
:depth: 2
```


A useful invoice program can read an order rather than require source edits for every customer. Console input arrives as text. The program must interpret that text, apply the calculation, and display a result with clear labels and units. This section uses known-valid numeric input; recovery from invalid input belongs in Chapters 4 and 6.


```{index} Console.ReadLine; Console.Write; end of input
```

(read-from-console)=

## Prompt, Read, and Keep the Text



A **prompt** tells the user what to enter. `Console.Write` prints without adding a newline; `Console.WriteLine` adds one. `Console.ReadLine()` reads a line without its line-ending characters. It returns `null` when no more input is available, which is different from the empty string returned for a blank line.[^1]

In the browser, click Run to reveal an Input field, enter the requested sample text, and run again or press Enter. In a local console, type the value when prompted. Each input-reading cell is independent.


In [1]:
// Sample input: Riverside Dental
Console.Write("Customer: ");
string customer = Console.ReadLine() ?? "";
Console.WriteLine();
Console.WriteLine($"Invoice for {customer}");


Customer: 
Invoice for Riverside Dental


The `??` operator uses its right-hand value only when the left-hand value is `null`. Here, it substitutes an empty string for absent input, avoiding a nullable assignment warning. It does not prove that a customer name is present: a blank line remains blank.

Browser and terminal interfaces may display input differently. The program does not itself echo the entered text; `Console.WriteLine()` adds a newline so the result is separate from the prompt even when redirected input is not echoed.


```{index} parsing; int.Parse; decimal.Parse; input contract
```

(parsing-input)=

## Parsing Text Is Different from Casting



`int.Parse("12")` interprets digit text as the integer 12. A numeric cast converts an existing numeric value, as in `(int)12.8`. Parsing is necessary because input is text, not because strings are immutable.

For this example, enter a whole-number quantity between zero and 100. That is an **input contract**, an assumption the calculation relies on. `int.Parse` rejects malformed text and values outside the `int` range; it does not enforce the narrower business range by itself.


In [1]:
// Sample input: 12
Console.Write("Quantity (whole number, 0 to 100): ");
string quantityText = Console.ReadLine() ?? "";
int quantity = int.Parse(quantityText);
Console.WriteLine();
Console.WriteLine($"Cases ordered: {quantity}");


Quantity (whole number, 0 to 100): 
Cases ordered: 12


`"twelve"`, `"12.5"`, and a blank line are not valid inputs for this `int.Parse` example. `"-1"` parses as an integer but violates the order contract. An EOF becomes an empty string here, which parsing rejects; it is not silently treated as quantity zero. Later chapters add branching and `TryParse` to accept or reject input gracefully.

An input contract describes valid data; validation checks whether actual data satisfies it. A program that merely states an assumption has not implemented validation.


```{index} CultureInfo; InvariantCulture; decimal separator
```

### Parsing Fractional Input and Culture



Number conventions vary: some locales use a decimal point and others a comma. This example explicitly uses `InvariantCulture` to interpret a period as the decimal separator and format output consistently. Its prompt states that convention. Other applications should select conventions appropriate to their users.

The `using System.Globalization;` directive makes the `CultureInfo` name available. Passing `CultureInfo.InvariantCulture` to `decimal.Parse` chooses the parsing convention; setting `CurrentCulture` chooses the default display convention used by this cell.


In [1]:
using System.Globalization;
CultureInfo.CurrentCulture = CultureInfo.InvariantCulture;

// Sample input: 38.50
Console.Write("Unit price (use a decimal point): ");
string priceText = Console.ReadLine() ?? "";
decimal unitPrice = decimal.Parse(priceText, CultureInfo.InvariantCulture);
Console.WriteLine();
Console.WriteLine($"Unit price: {unitPrice:F2} USD");


Unit price (use a decimal point): 
Unit price: 38.50 USD


`double.Parse` similarly reads an approximate measurement. Use `decimal.Parse` here because the data model requires decimal invoice amounts. This parsing overload can also accept some grouping and whitespace syntax; it is not a strict “digits plus one point” validator. Do not enter commas into this period-based example.

**Predict:** If a quantity and unit price are both read as strings, why can you not multiply them? First parse each into the intended numeric type.


In [ ]:
// ### Exercise: Read and Calculate a Subtotal
// Read one whole-number quantity (sample 3). Use unitPrice 10.10m and print the subtotal with two fractional places. Assume a nonnegative valid int input.
// Your code starts here.

// Your code ends here.


In [1]:
using System.Globalization;
CultureInfo.CurrentCulture = CultureInfo.InvariantCulture;

// ### Solution
Console.Write("Quantity: ");
string quantityText = Console.ReadLine() ?? "";
int quantity = int.Parse(quantityText);
decimal unitPrice = 10.10m;
decimal subtotal = quantity * unitPrice;
Console.WriteLine();
Console.WriteLine($"Subtotal: {subtotal:F2} USD");


Quantity: 
Subtotal: 30.30 USD


```{index} string interpolation; format specifier
```

(string-interpolation)=

## Interpolation Makes a Report Readable



Prefix a string with `$` and put expressions in braces to form an **interpolated string**. In `$"Cases: {quantity}"`, the value replaces `{quantity}`. A format suffix such as `:F2` displays two fractional places. It changes the representation of the value, not the stored value.


In [1]:
using System.Globalization;
CultureInfo.CurrentCulture = CultureInfo.InvariantCulture;

int quantity = 12;
decimal unitPrice = 38.50m;
decimal rawTax = 0.505m;
Console.WriteLine($"Order: {quantity} cases at {unitPrice:F2} USD each");
Console.WriteLine($"Tax displayed: {rawTax:F2}");
Console.WriteLine($"Tax stored: {rawTax}");


Order: 12 cases at 38.50 USD each
Tax displayed: 0.51
Tax stored: 0.505


The tax display is 0.51 while the stored value remains 0.505. Assign `Math.Round(...)` when the business calculation requires a rounded amount. Use a currency code or state the currency in the report; a currency display format does not convert exchange rates.


```{index} composite formatting; format item
```

(format-strings)=

### Composite Formatting and Escaped Braces



**Composite formatting** uses numbered placeholders and a separate argument list. Index zero selects the first argument after the format string. It can express the same report as interpolation, but requires tracking indexes.


In [1]:
using System.Globalization;
CultureInfo.CurrentCulture = CultureInfo.InvariantCulture;

int quantity = 12;
decimal price = 38.50m;
Console.WriteLine("Order: {0} cases at {1:F2} USD each", quantity, price);
Console.WriteLine($"Order: {quantity} cases at {price:F2} USD each");
Console.WriteLine("Reference {{{0:D4}}}", 7);


Order: 12 cases at 38.50 USD each
Order: 12 cases at 38.50 USD each
Reference {0007}


`D4` pads an integer with leading zeros to at least four digits; `D` is an integer decimal-digit format, not the `decimal` type. The final result is `Reference {0007}`. In a composite format string, doubled braces produce literal braces. A placeholder referring to a missing argument would fail at runtime.

The same placeholder rules apply to `string.Format`, which returns a formatted string instead of printing it.


```{index} Console.WriteLine; Console.Write; string.Format
```

(substitution-in-writeline)=

### Output Methods Have Different Roles



In [1]:
int quantity = 12;
string message = string.Format("Cases ordered: {0}", quantity);
Console.Write("Report: ");
Console.WriteLine(message);
Console.WriteLine("End of report");


Report: Cases ordered: 12
End of report


`string.Format` creates `message`; `Console.Write` prints the prefix; `Console.WriteLine` prints the message and ends that line. Keeping calculation separate from presentation makes both easier to inspect.


```{index} alignment; minimum field width; invoice table
```

(left-justification)=

## Align a Small Invoice Without Loops



An interpolation such as `{label,-12}` left-aligns text in a field at least 12 characters wide. `{amount,10:F2}` right-aligns a two-place number in a field at least 10 characters wide. Width is a **minimum**, not truncation: long text can extend beyond it. Tabs depend on display tab stops and do not guarantee consistent columns.

This chapter writes a few rows explicitly; Chapter 5 teaches loops for repeated rows. Include separators between columns so an unexpectedly long value does not merge invisibly with the next one.


In [1]:
using System.Globalization;
CultureInfo.CurrentCulture = CultureInfo.InvariantCulture;

decimal subtotal = 462.00m;
decimal tax = 32.34m;
decimal total = subtotal + tax;
Console.WriteLine($"{"Amount (USD)",-14} | {"Value",10}");
Console.WriteLine($"{"Subtotal",-14} | {subtotal,10:F2}");
Console.WriteLine($"{"Tax",-14} | {tax,10:F2}");
Console.WriteLine($"{"Total",-14} | {total,10:F2}");


Amount (USD)   |      Value
Subtotal       |     462.00
Tax            |      32.34
Total          |     494.34


**Review:** Confirm that the printed total agrees with the displayed components and that the currency is identified. A table can be neatly aligned and still communicate a wrong calculation.

For culture-dependent formatting, `amount.ToString("C2", culture)` uses the culture's currency convention. That convention determines a symbol and separators; it does not establish which currency a business amount represents. The invoice lab uses `USD` and fixed decimal-point formatting for reproducible classroom output.

**Exit check:** Describe the path from typed text through parsing, calculation, explicit rounding, and output formatting. Identify where invalid input could fail and where a business rule still needs validation.


```{index} downloadable invoice examples
```

### Saved Examples



Download {download}`Invoice.cs <../../materials/02/Invoice.cs>` for the lab's final report or {download}`ReadInvoice.cs <../../materials/02/ReadInvoice.cs>` for a quantity-driven calculation. Run each separately with `dotnet run Invoice.cs` or `dotnet run ReadInvoice.cs` using the .NET 10 SDK.


```{rubric} Footnotes
```

[^1]: Microsoft's [Console.ReadLine](https://learn.microsoft.com/en-us/dotnet/api/system.console.readline?view=net-10.0) reference distinguishes a line from end-of-input. [Standard numeric format strings](https://learn.microsoft.com/en-us/dotnet/standard/base-types/standard-numeric-format-strings) defines formats such as `D`, `F`, and `C`.
